In [4]:
import openai
import requests
import json


client = openai.OpenAI()

prompt = """ get_popular_movies(), get_movie_details(id), get_movie_credits(id) 이 세 함수는 영화 정보를 가져오는 함수야. 
https://nomad-movies.nomadcoders.workers.dev를 통해서 영화 정보를 가져올 수 있어. 
get_popular_movies()는 인기 영화 목록을 가져오고, 
get_movie_details(id)는 특정 영화의 상세 정보를 가져오고, 
get_movie_credits(id)는 특정 영화의 출연진 정보를 가져와. 이 함수들을 사용해서 인기 영화 목록을 가져와서 각 영화의 상세 정보와 출연진 정보를 출력해줘.
다음 중 질문을 보고 내가 어떤 함수를 실행하면 좋은지 함수와 파라미터를 알려줘.
다음 질문에 답해줘.
"지금 인기 있는 영화가 무엇인지 알려줘"
"movie ID 550에 해당하는 영화가 무엇인지 알려줘"
"movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"

"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        { "role": "user", 
         "content": prompt
         },
    ])

def get_popular_movies():
    response = requests.get("https://nomad-movies.nomadcoders.workers.dev/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits")
    return response.json()


for choice in response.choices:
    print(choice.message.content)

여기에서 각 질문에 대해 어떤 함수를 실행해야 할지와 필요한 파라미터를 정리해 드리겠습니다.

1. **질문:** "지금 인기 있는 영화가 무엇인지 알려줘"
   - **실행할 함수:** `get_popular_movies()`
   - **파라미터:** 없음

2. **질문:** "movie ID 550에 해당하는 영화가 무엇인지 알려줘"
   - **실행할 함수:** `get_movie_details(550)`
   - **파라미터:** `id=550`

3. **질문:** "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘"
   - **실행할 함수:** `get_movie_credits(550)`
   - **파라미터:** `id=550`

각 질문에 따라 필요한 함수를 호출하여 적절한 정보를 얻을 수 있습니다.


In [ ]:
import openai

client = openai.OpenAI()

messages = []

TOOLS  = {
    "type": "function",
    "function" : {
        "name": "get_weather",
        "description": "현재 날씨를 가져오는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "날씨를 알고 싶은 위치"
                }
            },
            "required": ["city"]
        }
    }
}

def ask_gpt(question):
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools = TOOLS,
        temperature=0.1,
    )
    answer = response.choices[0].message.content
    messages.append({"role": "assistant", "content": answer})
    return answer

while True:
    question = input("질문을 입력하세요 (종료하려면 'exit' 입력): ")
    
    if question.lower() == 'exit':
        break
    print("질문:", question)
    answer = ask_gpt(question)
    print("GPT의 답변:", answer)

KeyboardInterrupt: Interrupted by user

In [ ]:
import openai
import json
import requests

client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

messages = []

# ── Tools 정의 ──────────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "기준 영화 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]


# ── 실제 API 호출 함수 ─────────────────────────────────────
def get_popular_movies() -> str:
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)


def get_movie_details(id: int) -> str:
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)


def get_similar_movies(id: int) -> str:
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)


# ── 함수 매핑 ───────────────────────────────────────────────
TOOL_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}


# ── 메인 로직 ───────────────────────────────────────────────
def ask_gpt(question: str) -> str:
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
        temperature=0.1,
    )

    assistant_message = response.choices[0].message

    # tool_calls가 없을 때까지 반복 (연쇄 호출 대응)
    while assistant_message.tool_calls:
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            print(f"  🔧 {func_name}({func_args}) 호출 중...")

            try:
                result = TOOL_MAP[func_name](**func_args)
            except Exception as e:
                result = json.dumps({"error": str(e)})

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                }
            )

        # tool 결과를 포함하여 재호출
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            temperature=0.1,
        )
        assistant_message = response.choices[0].message

    answer = assistant_message.content
    messages.append({"role": "assistant", "content": answer})
    return answer


# ── 대화 루프 ───────────────────────────────────────────────
while True:
    question = input("\n질문을 입력하세요 (종료: exit): ")
    if question.lower() == "exit":
        break
    print("질문:", question)
    answer = ask_gpt(question)
    print("GPT:", answer)

질문: 지금 인기 있는 영화 알려줘
  🔧 get_popular_movies({}) 호출 중...
GPT: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Shelter**
   - 개봉일: 2026-01-28
   - 평점: 6.8
   - 개요: 한 남자가 외딴 섬에서 자발적으로 은둔 생활을 하다가 폭풍에서 어린 소녀를 구하면서 과거의 적들로부터 그녀를 보호하기 위해 은둔에서 나서게 됩니다.
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **The Bluff**
   - 개봉일: 2026-02-17
   - 평점: 6.0
   - 개요: 한 외딴 섬에서 평화로운 삶을 살던 전직 해적이 복수심에 불타는 전 선장의 귀환으로 인해 가족을 지키기 위해 과거의 피비린내 나는 사건과 맞서게 됩니다.
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

3. **Mercy**
   - 개봉일: 2026-01-20
   - 평점: 7.1
   - 개요: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받으며, 자신이 지지했던 AI 판사에게 자신의 무죄를 입증해야 하는 상황에 처합니다.
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

4. **Pose**
   - 개봉일: 2026-02-25
   - 평점: 7.7
   - 개요: 한 외딴 시골 저택에서 은둔하는 예술가가 전 연인, 집착하는 팬, 잠재적인 뮤즈와 함께 격렬하고 혼란스러운 주말을 보내며 예술적 영감을 찾으려 합니다.
   - ![Pose](https://image.tmdb.org/t/p/w780/qhyyyWrHUbl6QG4udAJj17CBa5.jpg)

5. **Hellfire**
   - 개봉일: 20